
# 🧭 CMPE 272 – In‑Class ERP Lab (Colab)  
**Topic:** ERP Systems — Order‑to‑Cash, Procure‑to‑Pay, Basic MRP, Data Quality, and Controls  
**Fictional Company:** **BlueFin eFoils, Inc.** 🏄‍♀️🔋  
**Team Size:** 2–4 students • **Time:** ~90–120 minutes

> **How to use this notebook**
> 1. Go to **Runtime → Run all** to install packages and generate the sample dataset.  
> 2. Complete every **`TODO`** you see (code and short written responses).  
> 3. Work as a team; rotate a *driver* (keyboard) every section.  
> 4. At the end, make sure your team’s names are in the header below and upload/submit as directed by your instructor.

**Team members:** Isaac Kim, Oleg Mrynski, Liliya Panfilova, and Jagath Jijo

---

### Learning goals
- See how core ERP processes tie together: **Sales/Distribution**, **Inventory**, **Procurement**, **Production (MRP)**, and **Finance**.  
- Implement small but realistic ERP routines: allocations, reorders, and simple BOM planning.  
- Practice **data quality** cleanup and **roles/controls** reasoning.  
- Communicate decisions succinctly with evidence from your data.




## 0) Quick setup — run this first
This cell installs what we need and generates a small, messy-but-fun demo dataset. If you re-run, it will reset to the original state.


In [1]:

# @title Install & Imports (Run me)
# Minimal deps so the whole class can run quickly.
import pandas as pd
import numpy as np
import datetime as dt
import math
import textwrap
from collections import defaultdict

# Optional plotting (use only if you want): matplotlib is available by default in Colab.
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Random seed for reproducibility across teams
rng = np.random.default_rng(272)

# -------------------------------
# Scenario: BlueFin eFoils, Inc.
# -------------------------------
# BlueFin manufactures and sells electric hydrofoil surfboards (eFoils).
# Products:
#  - BF-100 Board Kit (finished good)
#  - BAT-9 Battery Pack
#  - CTRL-2 Control Unit
#  - FOIL-3 Hydrofoil Assembly
#  - CHR-1 Charger
# The BF-100 is assembled from BAT-9, CTRL-2, FOIL-3, and CHR-1.
# Sales spike is expected next month due to a viral video. Inventory is tight.

today = pd.Timestamp.today().normalize()

products = pd.DataFrame([
    {"sku":"BF-100","name":"BlueFin eFoil Board Kit","uom":"ea","unitCost":1800.00,"unitPrice":2999.00,"leadTimeDays":7,"reorderPoint":12,"reorderQty":20},
    {"sku":"BAT-9","name":"Battery Pack 9kWh","uom":"ea","unitCost":750.00,"unitPrice":999.00,"leadTimeDays":10,"reorderPoint":25,"reorderQty":40},
    {"sku":"CTRL-2","name":"Control Unit Mk2","uom":"ea","unitCost":220.00,"unitPrice":399.00,"leadTimeDays":8,"reorderPoint":25,"reorderQty":40},
    {"sku":"FOIL-3","name":"Hydrofoil Assembly v3","uom":"ea","unitCost":380.00,"unitPrice":599.00,"leadTimeDays":9,"reorderPoint":25,"reorderQty":40},
    {"sku":"CHR-1","name":"Fast Charger 2kW","uom":"ea","unitCost":95.00,"unitPrice":179.00,"leadTimeDays":5,"reorderPoint":30,"reorderQty":60},
])

inventory = pd.DataFrame([
    {"sku":"BF-100","warehouse":"SJ-01","onHand":14,"reserved":0,"inTransit":0},
    {"sku":"BAT-9","warehouse":"SJ-01","onHand":28,"reserved":0,"inTransit":10},
    {"sku":"CTRL-2","warehouse":"SJ-01","onHand":19,"reserved":0,"inTransit":0},
    {"sku":"FOIL-3","warehouse":"SJ-01","onHand":26,"reserved":0,"inTransit":0},
    {"sku":"CHR-1","warehouse":"SJ-01","onHand":32,"reserved":0,"inTransit":0},
])

# Simple BOM (one level for speed)
bom = pd.DataFrame([
    {"parent":"BF-100","component":"BAT-9","qtyPer":1},
    {"parent":"BF-100","component":"CTRL-2","qtyPer":1},
    {"parent":"BF-100","component":"FOIL-3","qtyPer":1},
    {"parent":"BF-100","component":"CHR-1","qtyPer":1},
])

# Customers
customers = pd.DataFrame([
    {"customerId":"C100","name":"Sunny Surf Co.","terms":"NET30"},
    {"customerId":"C200","name":"Wave Riders LLC","terms":"NET30"},
    {"customerId":"C300","name":"Direct-to-Consumer","terms":"CC"},
])

# Sample orders for the next two weeks (some will exceed stock)
orderLines = pd.DataFrame([
    {"orderId":"SO-001","orderDate":today + pd.Timedelta(days=1),"customerId":"C100","sku":"BF-100","qty":8},
    {"orderId":"SO-002","orderDate":today + pd.Timedelta(days=2),"customerId":"C200","sku":"BF-100","qty":7},
    {"orderId":"SO-003","orderDate":today + pd.Timedelta(days=2),"customerId":"C300","sku":"BF-100","qty":6},
    {"orderId":"SO-004","orderDate":today + pd.Timedelta(days=3),"customerId":"C100","sku":"BF-100","qty":9},
])

# Vendors (with messy/duplicate data on purpose)
vendors = pd.DataFrame([
    {"vendorId":"V01","name":"Voltix Power","phone":"(408) 555-0100","email":"orders@voltix.com","defaultSku":"BAT-9"},
    {"vendorId":"V01X","name":"Voltix  Power ","phone":"408-555-0100","email":"sales@voltix.com","defaultSku":"BAT-9"},
    {"vendorId":"V02","name":"FoilWorks","phone":"+1 408.555.0200","email":"contact@foilworks.io","defaultSku":"FOIL-3"},
    {"vendorId":"V03","name":"Controllytics","phone":"4085550300","email":"hello@controllytics.ai","defaultSku":"CTRL-2"},
    {"vendorId":"V04","name":"ChargeFast","phone":"(408)-555-0400","email":"orders@chargefast.co","defaultSku":"CHR-1"},
])

leadTimes = dict(zip(products["sku"], products["leadTimeDays"]))

# Simple GL (very lightweight for the exercise)
glAccounts = pd.DataFrame([
    {"account":"1200","name":"Accounts Receivable"},
    {"account":"1400","name":"Inventory"},
    {"account":"2000","name":"Accounts Payable"},
    {"account":"4000","name":"Revenue"},
    {"account":"5000","name":"COGS"},
])

print("✅ Dataset loaded. Try: products.head(), inventory, orderLines, vendors, bom")


✅ Dataset loaded. Try: products.head(), inventory, orderLines, vendors, bom


In [2]:
bom.head()

,parent,component,qtyPer
0,BF-100,BAT-9,1
1,BF-100,CTRL-2,1
2,BF-100,FOIL-3,1
3,BF-100,CHR-1,1



## 1) Company overview & quick process mapping (10–15 min)

**BlueFin eFoils, Inc.** designs and manufactures electric hydrofoil surfboards. They sell direct and to specialty surf dealers. A viral video just boosted demand for the **BF‑100** finished good, and the team is worried about **stockouts** and **component shortages**.

**Your task:** In teams, make a lean mapping of the two big flows:

- **Order‑to‑Cash (O2C):** From Sales Order → Allocation → Pick/Pack/Ship → Invoice → Cash Receipt → GL postings.  
- **Procure‑to‑Pay (P2P):** From Requisition → PO → Receiving → AP Invoice → Payment → GL postings.

### ✅ TODO (write below this cell)
1. Identify the **ERP modules** involved in O2C and P2P and list **key master data** used by each (customers, vendors, items, price lists, etc.).  
2. Write **2–3 key risks** for each process (e.g., wrong shipments, duplicate payments, etc.) and a **control** that mitigates each risk.
3. Sketch a simple text map of O2C and P2P (you can use arrows like `->`).

> *Tip:* Keep it short but specific. You’ll refer back to this in later steps.


## Order‑to‑Cash

#### ERP modules involved in O2C:
- Sales and Distribution
- Customer Relationship Management (CRM)
- Inventory Management
- Warehouse Management
- Accounts Receivable
- Finance / Accounting
- Master Data Management


#### Key master data used by O2C:
- Customer name and address
- Customer payment terms
- Product id and name
- Product sales price
- Tax information
- Warehouse bin (product location)
- Shipping information

#### Risks involved in O2C:
- Incorrect price. Customer makes an order, when price in the system is lower than required, the invoice gets sent and nobody notices a mistake.
- Fulfillment error. The warehouse picks incorrect product and ships to customer.

#### Controls:
- Incorrect price. The ERP system should use an approved price list and restrict manual price changes. If the entered price differs from the standard price beyond an allowed tolerance, the order should require approval before the invoice can be generated.
- Fulfillment error.  Fulfillment personnel should scan the product barcode while picking or packing the order. If the scanned product does not match the SKU on the sales order, the ERP system should block the next step, such as generating shipping documents or confirming shipment.

#### Text map:
Customer Order -> Sales Order Created -> Inventory Allocated -> Warehouse Picks/Packs/Ships -> Customer Invoice Created -> Payment Received -> GL Updated

## Procure‑to‑Pay

#### ERP modules involved in P2P:

- Supplier Relationship Management
- Inventory Management
- Materials Management / Procurement
- Warehouse Management
- Master Data Management
- Accounts Payable
- Finance / Accounting

#### Key master data used by P2P:

- Supplier(s) id, name and address
- Supplier payment terms
- Product id and name
- Product purchase price
- Warehouse bin (product location)

#### Risks involved in P2P:
- Incorrect Invoice. The supplier sends an invoice with a higher price than expected. The purchasing personnel do not notice the difference and pay the amount stated on the invoice. This may happen, for example, if the supplier announced a price increase, changed the price in their internal system, and incorrectly applied it to an order that had already been shipped.
- Wrong quantity. The warehouse personnel incorrectly count the quantity while receiving the product. As a result, the inventory information becomes incorrect, which may also lead to an incorrect invoice amount being approved.

#### Controls:
- Incorrect Invoice. The ERP system should not allow an invoice to be paid if the price does not match the price in the system, unless it is explicitly approved by a manager.
- Wrong quantity. The warehouse personnel should scan each box they receive so that the system records the actual quantity received. The system should then compare the received quantity with the purchase order and flag any differences, such as receiving more or fewer items than were ordered.
#### Text map:
Need Identified -> Requisition -> Purchase Order -> Goods Received -> Supplier Invoice -> Payment -> GL Updated


## 2) Mini-sim: Order‑to‑Cash – allocate and invoice (20–25 min)

You’ll implement a tiny allocation & invoicing routine for **BF‑100** using the current `inventory` and `orderLines`.

**Business rules:**
- Allocate finished goods first‑come‑first‑served by `orderDate`.
- If you can’t fully allocate, record the **backorder** quantity.
- When allocated, create a simple **invoice** with `unitPrice` from `products` and compute line `amount`.

Below is a scaffold with `TODO` markers.


In [3]:

# Helper: get product price and cost quickly
def getProductRecord(sku: str) -> pd.Series:
    rec = products.loc[products["sku"] == sku]
    return rec.iloc[0] if not rec.empty else None

def allocateInventory(orderLinesDf: pd.DataFrame, inventoryDf: pd.DataFrame) -> pd.DataFrame:
    """
    Allocate on-hand inventory to orders by orderDate (FIFO).
    Returns a DataFrame with columns:
      orderId, orderDate, customerId, sku, qty, allocatedQty, backorderedQty
    """
    # TODO: Implement allocation logic
    # Steps:
    # 1) Sort orders by orderDate, then orderId (stable)
    # 2) Track available on-hand by sku (single warehouse for simplicity)
    # 3) For each order line, allocate up to available; compute backordered remainder
    # 4) Update the available bucket
    # 5) Return a DataFrame with allocated/backordered columns filled
    sortedLines = orderLinesDf.sort_values(["orderDate", "orderId"]).copy()
    sortedLines["allocatedQty"] = 0
    sortedLines["backorderedQty"] = sortedLines["qty"]  # default until allocated

    # >>> YOUR CODE HERE <<<

    # available inventory
    availableInventory = {}

    for _, row in inventoryDf.iterrows():
        availableInventory[row["sku"]] = int(row["onHand"])

    # allocating inventory for every order
    for index, row in sortedLines.iterrows():

        sku = row["sku"]
        requestedQty = int(row["qty"])
        availableQty = availableInventory.get(sku, 0)
        allocatedQty = min(requestedQty, availableQty)
        backorderedQty = requestedQty - allocatedQty

        # update sortedLines
        sortedLines.loc[index, "allocatedQty"] = allocatedQty
        sortedLines.loc[index, "backorderedQty"] = backorderedQty

        # update inventory
        availableInventory[sku] = availableQty - allocatedQty

    return sortedLines

def createInvoices(allocationsDf: pd.DataFrame) -> pd.DataFrame:
    """
    Create invoices for lines with allocatedQty > 0.
    Columns: invoiceId, orderId, customerId, sku, allocatedQty, unitPrice, amount, invoiceDate, terms
    """
    # TODO:
    # 1) Filter allocated lines
    # 2) Lookup price and customer terms
    # 3) Compute amount = allocatedQty * unitPrice
    # 4) Generate invoiceId like INV-001, INV-002,...
    invRows = []

    # >>> YOUR CODE HERE <<<
    allocatedLines = allocationsDf.loc[allocationsDf["allocatedQty"] > 0].copy()

    for i, (_, row) in enumerate(allocatedLines.iterrows(), start=1):
        sku = row["sku"]
        customerId = row["customerId"]
        allocatedQty = row["allocatedQty"]

        # product price
        productRecord = getProductRecord(sku)
        unitPrice = productRecord["unitPrice"]

        # customer payment terms
        customerRecord = customers.loc[customers["customerId"] == customerId].iloc[0]
        terms = customerRecord["terms"]

        # invoice amount
        amount = allocatedQty * unitPrice
        invoiceId = f"INV-{i:03d}"

        # invouce row
        invRows.append({
            "invoiceId": invoiceId,
            "orderId": row["orderId"],
            "customerId": customerId,
            "sku": sku,
            "allocatedQty": allocatedQty,
            "unitPrice": unitPrice,
            "amount": amount,
            "invoiceDate": today,
            "terms": terms
        })


    invoicesDf = pd.DataFrame(invRows, columns=["invoiceId","orderId","customerId","sku","allocatedQty","unitPrice","amount","invoiceDate","terms"])
    return invoicesDf

# --- Quick test harness (you can run after implementing) ---
allocations = allocateInventory(orderLines, inventory)
invoices = createInvoices(allocations)
invoices.head()
allocations.head()

,orderId,orderDate,customerId,sku,qty,allocatedQty,backorderedQty
0,SO-001,2026-09-29,C100,BF-100,8,8,0
1,SO-002,2026-09-30,C200,BF-100,7,6,1
2,SO-003,2026-09-30,C300,BF-100,6,0,6
3,SO-004,2026-10-01,C100,BF-100,9,0,9



## 3) Procure‑to‑Pay – smart reorders & receipts (20–25 min)

We’ll add a small **reorder suggestion** that generates POs when stock is below `reorderPoint`. Then we’ll **receive** those POs to update inventory.

**Business rules:**
- For each `sku`, compute **available** = `onHand + inTransit - reserved`.
- If `available < reorderPoint`, propose a PO for `reorderQty`.
- On receiving a PO, move qty from `inTransit` to `onHand`.

Then consider: *How would you reflect AP and cash movement in GL at a high level?* (Write short notes in markdown after the code.)

Below is a scaffold.


In [4]:
def suggestReorders(inventoryDf: pd.DataFrame, productsDf: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a DataFrame of suggested POs with columns:
      poId, sku, qty, etaDate, vendorGuess
    Vendor is guessed from `vendors.defaultSku` (simplified!)
    """
    # TODO:
    # 1) For each sku, compute available
    # 2) If below reorderPoint, propose reorderQty and ETA today + leadTimeDays
    # 3) Guess vendor by matching vendors.defaultSku == sku (first hit)
    poRows = []
    poCounter = 1

    # >>> YOUR CODE HERE <<<
    # iterate over products and pull off relevant columns
    for _, prod in productsDf.iterrows():
        sku = prod["sku"]
        reorderPoint = prod["reorderPoint"]
        reorderQty = prod["reorderQty"]
        leadDays = prod["leadTimeDays"]

        # get the matching inventory row by matching on "sku"
        invRow = inventoryDf[inventoryDf["sku"] == sku]

        # if the row isn't empty, calculate available stock
        if not invRow.empty:
            onHand = invRow["onHand"].values[0]
            inTransit = invRow["inTransit"].values[0]
            reserved = invRow["reserved"].values[0]
            available = onHand + inTransit - reserved
        # if the row is empty, the available stock is 0
        else:
            available = 0

        # "main logic" - check if the available stock is below
        # reorder point.
        if available < reorderPoint:
            poId = f"PO-{poCounter:03d}"
            etaDate = today + pd.Timedelta(days=int(leadDays))


            matchVendor = vendors.loc[vendors["defaultSku"] == sku]
            vendorGuess = matchVendor["name"].iloc[0] if not matchVendor.empty else "Unknown"

            poRows.append({
                "poId": poId,
                "sku": sku,
                "qty": reorderQty,
                "etaDate": etaDate,
                "vendorGuess": vendorGuess
            })
            poCounter += 1

    return pd.DataFrame(poRows, columns=["poId", "sku", "qty", "etaDate", "vendorGuess"])

def receivePo(inventoryDf: pd.DataFrame, poDf: pd.DataFrame) -> pd.DataFrame:
    """
    Simulate receiving all PO lines today.
    Moves qty from inTransit to onHand for matching sku at SJ-01.
    Returns an updated copy of inventoryDf.
    """
    inv = inventoryDf.copy()
    # TODO: implement simple receipt logic
    # >>> YOUR CODE HERE <<<

    # loop through each PO in poDf
    for _, row in poDf.iterrows():
        sku = row["sku"]
        received_qty = row["qty"]

        # find the matching inventory row's index:
        #   inv["sku"] must match the poDf's row's "sku"
        #   AND inv["warehouse"] needs to match "SJ-01"
        inv_row_idx = inv.loc[
            (inv["sku"] == sku) &
             (inv["warehouse"] == "SJ-01")
             ].index

        if not inv_row_idx.empty:
            # Increase onHand by the received quantity
            inv.loc[inv_row_idx, "onHand"] += received_qty

            # Decrease inTransit, but not below zero.
            current_in_transit = inv.loc[inv_row_idx, "inTransit"].values[0]
            decrease_in_transit_by = min(received_qty, current_in_transit)
            inv.loc[inv_row_idx, "inTransit"] -= decrease_in_transit_by
            # Ensure inTransit doesn't go below 0
            inv.loc[inv_row_idx, "inTransit"] = inv.loc[inv_row_idx, "inTransit"].clip(lower=0)


    return inv

# --- Quick test harness (run after you implement) ---
pos = suggestReorders(inventory, products)
pos.head()
inventory_after = receivePo(inventory, pos)
inventory_after

,sku,warehouse,onHand,reserved,inTransit
0,BF-100,SJ-01,14,0,0
1,BAT-9,SJ-01,28,0,10
2,CTRL-2,SJ-01,59,0,0
3,FOIL-3,SJ-01,26,0,0
4,CHR-1,SJ-01,32,0,0


***How would you reflect AP and cash movement in GL at a high level? (Write short notes in markdown after the code.)***

When the items arrive, BlueFin adds their value to the Inventory.
 - At this point, the suppleier has not been paid yet so it also records a temporary amount owed.

When the supplier sends an invoice, the temporary amount is moved into AP.
 - AP is the amount of money BlueFin owes the supplier.

Once BlueFin pays the suppler, AP goes down because the debt is paid and Cash goes down as well.

Overall:
 1. Goods received: increase inventory, record temporary liability.
 2. Invoice sent + approved: move the temporary liability to AP.
 3. Supplier paid: decrease AP and decrease Cash.


## 4) Tiny MRP: explode BOM and plan components (15–20 min)

A dealer asked for **N units of BF‑100** next week. You’ll compute **gross requirements** for components and suggest **planned order releases** by lead time.

**Business rules (one-level BOM for speed):**
- Gross requirements for each component = demand for BF‑100 × `qtyPer` in `bom`.
- If component `available` (onHand + inTransit - reserved) is insufficient, suggest a planned release for the shortfall.
- Use `leadTimeDays` per `products` to set a **releaseDate** = `needDate - leadTimeDays` (assume needDate is next Friday).

Fill in the `TODO` below.


In [5]:

def planMrpForBf100(demandQty: int, needDate: pd.Timestamp) -> pd.DataFrame:
    """
    Returns a DataFrame with columns:
      component, grossReq, available, shortage, plannedReleaseQty, releaseDate
    """
    rows = []
    for _, r in bom[bom["parent"]=="BF-100"].iterrows():
        comp = r["component"]
        qtyPer = r["qtyPer"]
        gross = demandQty * qtyPer

        # Compute available
        invRow = inventory[(inventory["sku"]==comp) & (inventory["warehouse"]=="SJ-01")]
        available = int(invRow["onHand"].iloc[0] + invRow["inTransit"].iloc[0] - invRow["reserved"].iloc[0])

        # TODO: compute shortage and planned release by comparing to gross
        # >>> YOUR CODE HERE <<<
        shortage = max(0, gross - available)
        plannedReleaseQty = shortage

        # compute releaseDate = needDate - leadTimeDays
        lt = leadTimes.get(comp, 0)
        releaseDate = (needDate - pd.Timedelta(days=int(lt))).normalize()

        rows.append({
            "component": comp,
            "grossReq": gross,
            "available": available,
            "shortage": shortage,
            "plannedReleaseQty": plannedReleaseQty,
            "releaseDate": releaseDate,
        })
    return pd.DataFrame(rows)

# --- Quick test harness ---
next_friday = (pd.Timestamp.today() + pd.offsets.Week(weekday=4)).normalize()
mrp = planMrpForBf100(demandQty=15, needDate=next_friday)
mrp


,component,grossReq,available,shortage,plannedReleaseQty,releaseDate
0,BAT-9,15,38,0,0,2026-09-22
1,CTRL-2,15,19,0,0,2026-09-24
2,FOIL-3,15,26,0,0,2026-09-23
3,CHR-1,15,32,0,0,2026-09-27



## 5) Data quality & governance (15–20 min)

The `vendors` master has **duplicates**, weird spacing, and **inconsistent phone formats**. Clean it!

**Success criteria:**
- Trim whitespace and normalize case where helpful.
- Standardize phone to E.164-lite: `+14085550100` style.
- Deduplicate records such that **"Voltix Power"** variants become a single vendor row (keep the most complete info).
- Explain your **dedupe key** and **merge logic** in a few bullets under the cell.

Fill in the `TODO` below.


In [6]:
def normalizePhone(phone: str) -> str:
    # Example normalizer to help you get started
    digits = "".join(ch for ch in str(phone) if ch.isdigit())
    if digits.startswith("1") and len(digits) == 11:
        return "+" + digits
    if len(digits) == 10:
        return "+1" + digits
    # fallback (you can improve this)
    return "+" + digits if digits else ""

def cleanVendorMaster(vendorsDf: pd.DataFrame) -> pd.DataFrame:
    """
    Return a cleaned and de-duplicated vendor master.
    Hints:
      - .str.strip(), .str.title()
      - apply(normalizePhone)
      - groupby on a dedupe key you design
      - keep the "best" email/phone with the most characters
    """
    cleaned = vendorsDf.copy()

    # TODO: implement cleaning + dedupe
    # >>> YOUR CODE HERE <<<
    cleaned["name"] = cleaned["name"].apply(lambda x: " ".join(str(x).split()).strip().title())
    cleaned["email"] = cleaned["email"].str.strip().str.lower()
    cleaned["phone"] = cleaned["phone"].apply(normalizePhone)

    cleaned["dedupeKey"] = cleaned["name"]

    def longestValue(series):
        values = series.dropna().astype(str)
        return max(values, key=len) if not values.empty else ""

    mergedRows = []

    for _, group in cleaned.groupby("dedupeKey", sort=False):
        merged = group.iloc[0].copy()
        merged["phone"] = longestValue(group["phone"])
        merged["email"] = longestValue(group["email"])
        mergedRows.append(merged)

    cleaned = pd.DataFrame(mergedRows).drop(columns=["dedupeKey"])

    return cleaned


# --- Quick test harness ---
vendors_clean = cleanVendorMaster(vendors)
vendors_clean

,vendorId,name,phone,email,defaultSku
0,V01,Voltix Power,+14085550100,orders@voltix.com,BAT-9
2,V02,Foilworks,+14085550200,contact@foilworks.io,FOIL-3
3,V03,Controllytics,+14085550300,hello@controllytics.ai,CTRL-2
4,V04,Chargefast,+14085550400,orders@chargefast.co,CHR-1


***Explain your dedupe key and merge logic in a few bullets under the cell.***

Dedupe Key: we used the normalized vendor names as the dedupe key.
 - An edge case was making sure that internal whitespaces were removed as well as trailing whitespaces.
 - The edge case was Voltix Power - there was an internal whitespace that was not getting removed which resulted in two records with the same name, "Voltix Power." The lambda function is what ensures that the vendor name is properly normalized.

Merge Logic: we grouped records by the normalized vendor name and retained the first record's vendor ID and default SKU.


## 6) Roles & controls: Who can do what? (10–15 min)

Suppose BlueFin uses these **roles** (SoD = Segregation of Duties):

- **SalesRep:** create orders, view inventory.
- **Warehouse:** allocate, ship goods.
- **Buyer:** create POs, receive goods.
- **APClerk:** enter vendor invoices, schedule payments.
- **Controller:** approve payments, post period close.

**Risks to avoid:** one person both **creates** and **pays** a vendor invoice; one person **creates** and **ships** a sales order.

**TODO:** Implement a simple access check and identify SoD conflicts.


In [12]:

roles = {
    "SalesRep": {"createSO","viewInv"},
    "Warehouse": {"allocate","ship","viewInv"},
    "Buyer": {"createPO","receive","viewInv"},
    "APClerk": {"enterAP","schedulePay"},
    "Controller": {"approvePay","postClose","viewInv"},
}

sodConflicts = [
    {"actions":{"enterAP","approvePay"}, "reason":"Same person could create and approve vendor payments."},
    {"actions":{"createSO","ship"}, "reason":"Same person could create and ship a fraudulent order."},
]

def canPerform(role: str, action: str) -> bool:
    """Return True if role includes the action."""
    # TODO: implement check
    if role not in roles:
        return False
    return action in roles[role]

def hasConflict(userRoles: set) -> list:
    """
    Given a set of roles assigned to a user, return a list of conflict reasons matched.
    A conflict matches if the union of actions across those roles covers all actions in a conflict rule.
    """
    # TODO: implement SoD conflict detection
    allActions = set()
    for role in userRoles:
        if role in roles:
            allActions.update(roles[role])
    out = []
    for conflict in sodConflicts:
        if conflict["actions"].issubset(allActions):
            out.append(conflict["reason"])
    return out

# --- Quick tests ---
print(canPerform("SalesRep","viewInv"))   # expect True
print(canPerform("SalesRep","ship"))      # expect False
print(hasConflict({"APClerk","Controller"}))  # expect one conflict


True
False
['Same person could create and approve vendor payments.']



## ⭐ Optional (if time): KPI mini‑plot (10 min)

Create a simple **daily fulfillment KPI**: for each day in `orderLines`, after running your allocation, compute how many units were **fulfilled** vs **backordered**. Plot a single chart with two lines.

**Hint:** Use `matplotlib` (no special styles/colors). Keep it simple.


In [ ]:
plt.figure()
daily = allocations.groupby(allocations["orderDate"].dt.normalize())[["allocatedQty","backorderedQty"]].sum()
daily.plot()
plt.title("Daily BF-100: Allocated vs Backordered")
plt.xlabel("Date")
plt.ylabel("Units")
plt.show()

: 


## 7) Wrap‑up & quick reflection (5–10 min)
**As a team**, answer briefly:

1. Which **bottleneck** did you observe and what data would you collect next to improve it?  

2. If demand keeps spiking, which **policy change** (reorder points/qty, lead time buffers, safety stock) would you try first, and why?  
3. What **dashboard tile** (single KPI) would you pitch to leadership for next week? Include a one‑sentence rationale.


1.  From the sim, BF-100 finished goods seemed to run out fastest, we started with only 14 on hand and by the third order it couldn't allocate anything (SO-003 got 0). It'd probably help to look at more actual sales history and forecast data instead of just these 4 sample orders, so we can tell if this is a one time thing or a pattern.

2. Maybe start by raising the reorder point and safety stock for BF-100 since that's the item running out first and it has a 7 day lead time. Raising reorder qty for the components could help too since they're needed to build more boards, but BF-100 itself seems like the more immediate problem.

3. Something like "Days of Supply Remaining" for BF-100, on hand divided by average daily demand. Feels like a simple number that could give leadership a quick idea of how close we are to running out.


<details>
<summary><strong>Hints (expand if stuck)</strong></summary>

**Section 2 (Allocation):**
- Use a dict to track available stock by SKU: `available = {"BF-100": onHand}`.
- Allocation per line: `alloc = min(requested, available[sku])`; backorder is the rest.
- Don’t forget to subtract the allocation from `available`.

**Section 3 (Reorders):**
- Available = onHand + inTransit - reserved.
- Match vendor: `vendors.loc[vendors["defaultSku"]==sku].head(1)`.

**Section 4 (MRP):**
- Shortage = `max(0, grossReq - available)`; planned release = shortage (no lot sizing here).
- `releaseDate = needDate - pd.Timedelta(days=leadTimeDays)`.

**Section 5 (Data Quality):**
- Normalize name: `str.strip()`, squash multiple spaces: `" ".join(name.split())`.
- Choose longest non-empty email/phone when merging duplicates.

**Section 6 (Roles & Controls):**
- `canPerform` checks if `action in roles.get(role,set())`.
- For `hasConflict`, first union all actions across the user's roles, then check each conflict rule.

</details>



---

### ✅ What to submit
- This completed notebook with your **names at the top**.
- All **`TODO`** code sections implemented and short answers written.
- If you did the optional KPI, include the chart cell output.

> **Stretch idea:** Fork a copy and generalize the allocation to a multi‑SKU, multi‑warehouse scenario using the same approach.
